# Explainability And Export

This notebook regenerates feature-importance, SHAP outputs, susceptibility exports, and the conference paper DOCX.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config


In [ ]:
import joblib
import pandas as pd
from src.data_loader import load_raw_data, build_dataset_insight_tables
from src.preprocessing import prepare_dataset
from src.explainability import compute_tree_shap_outputs, save_feature_importance
from src.export_outputs import create_conference_docx, export_susceptibility_with_coordinates

raw_frame = load_raw_data(config.RAW_DATA_PATH)
insights = build_dataset_insight_tables(raw_frame)
prepared = prepare_dataset(
    raw_frame=raw_frame,
    target_column=config.TARGET_COLUMN,
    coordinate_columns=config.COORDINATE_COLUMNS,
    categorical_columns=config.CATEGORICAL_COLUMNS,
    cyclical_angle_columns=config.CYCLICAL_ANGLE_COLUMNS,
    spatial_block_size=config.SPATIAL_BLOCK_SIZE,
    spatial_folds=config.SPATIAL_FOLDS,
    validation_fold=config.VALIDATION_FOLD,
    test_fold=config.TEST_FOLD,
    seed=config.SEED,
)
primary_model = joblib.load(config.MODELS_DIR / 'xgboost.joblib')
benchmark_model = joblib.load(config.MODELS_DIR / 'random_forest.joblib')


In [ ]:
save_feature_importance(
    primary_model,
    prepared.feature_names,
    config.SHAP_DIR / 'xgboost_feature_importance.csv',
    config.SHAP_DIR / 'xgboost_feature_importance.png',
    'XGBoost Feature Importance',
    top_n=15,
)
compute_tree_shap_outputs(
    primary_model,
    prepared.test.features,
    config.SHAP_DIR,
    'xgboost',
    'XGBoost',
    max_samples=512,
    seed=config.SEED,
)


In [ ]:
import json
study_summary = json.loads((config.METRICS_DIR / 'study_summary.json').read_text(encoding='utf-8'))
test_metrics = pd.read_csv(config.METRICS_DIR / 'test_model_metrics.csv')
validation_metrics = pd.read_csv(config.METRICS_DIR / 'validation_model_metrics.csv')
create_conference_docx(
    summary_frame=insights['summary'],
    test_metrics_frame=test_metrics,
    validation_metrics_frame=validation_metrics,
    docs_dir=config.DOCS_DIR,
    figures_dir=config.FIGURES_DIR,
)


In [ ]:
export_susceptibility_with_coordinates(
    model=primary_model,
    feature_frame=prepared.full.features,
    metadata=prepared.full.metadata,
    threshold=float(study_summary['primary_model_test_metrics']['threshold']),
    model_name=config.PRIMARY_MODEL_NAME,
    file_path=config.PROCESSED_DATA_DIR / 'susceptibility_with_coordinates.csv',
)
